# Lakeflow SDP Pipelines

Welcome! This demo will teach you how to build declarative data pipelines using **Lakeflow Spark Declarative Pipelines (SDP)**.

---

## What You'll Learn

By the end of this demo, you will be able to:

1. Understand Lakeflow SDP concepts
2. Differentiate between streaming tables and materialized views
3. Create bronze layer with streaming tables
4. Build silver layer with data quality transformations
5. Create gold layer with materialized views
6. Configure and run an SDP pipeline
7. View pipeline results and lineage
8. Understand declarative vs imperative pipelines

---

## What is Lakeflow SDP?

**Lakeflow Spark Declarative Pipelines (SDP)** is a declarative framework for building reliable batch and streaming data pipelines. SDP extends and is interoperable with Apache Spark Declarative Pipelines, while running on the performance-optimized Databricks Runtime.

> **Note:** SDP was formerly known as Delta Live Tables (DLT). No migration is required — your existing code still works, but Databricks recommends adopting the new `pyspark.pipelines` module.

**Key concepts:**
* **Declarative** - Define WHAT you want, not HOW to do it
* **Automatic** - Handles dependencies, retries, scaling
* **Reliable** - Built-in data quality and monitoring
* **Incremental** - Processes only new data
* **Managed** - No infrastructure management

**Traditional pipeline vs SDP:**

| Traditional | SDP |
|-------------|-----|
| Write orchestration code | Declare tables |
| Handle dependencies manually | Automatic dependency resolution |
| Manage incremental logic | Built-in incremental processing |
| Write error handling | Automatic retries |
| Monitor manually | Built-in monitoring |

---

## Demo Scenario: Wanderbricks Analytics

You'll build a pipeline for **Wanderbricks** booking data:

**Pipeline flow:**
```
Raw Data (samples.wanderbricks)
    ↓
Bronze: Streaming tables (raw ingestion)
    ↓
Silver: Streaming tables (cleaned, validated)
    ↓
Gold: Materialized views (aggregated metrics)
```

**Data sources:**
* Bookings data
* Properties data
* Reviews data

---

**Let's get started!**

## Part 1: Lakeflow SDP Core Concepts

Before building, let's understand the key concepts.

---

### Streaming Tables vs Materialized Views

**SDP has two table types:**

---

### **1. Streaming Tables**

**What they are:**
* Process data incrementally as it arrives
* Append-only by default
* Maintain state for incremental processing
* Always up-to-date

**When to use:**
* Bronze layer (raw data ingestion)
* Silver layer (incremental transformations)
* Real-time or near-real-time data
* Append-only data (logs, events, transactions)
* When you need low latency

**Syntax:**
```python
from pyspark import pipelines as dp

@dp.table
def bronze_bookings():
    return spark.readStream.table("samples.wanderbricks.bookings")
```

---

### **2. Materialized Views**

**What they are:**
* Recompute entire result on each update
* Can handle updates and deletes
* Batch processing
* Optimized for aggregations

**When to use:**
* Gold layer (aggregations)
* Complex joins
* Aggregations (GROUP BY)
* When you need complete refresh
* Smaller result sets

**Syntax:**
```python
@dp.materialized_view
def gold_daily_revenue():
    return spark.read.table("silver_bookings").groupBy("date").agg(...)
```

---

### **Key Differences:**

| Feature | Streaming Table | Materialized View |
|---------|----------------|-------------------|
| **Decorator** | `@dp.table` | `@dp.materialized_view` |
| **Processing** | Incremental | Full refresh |
| **Read method** | `spark.readStream.table()` | `spark.read.table()` |
| **Updates** | Append-only | Supports updates/deletes |
| **Latency** | Low (real-time) | Higher (batch) |
| **Use case** | Bronze, Silver | Gold (aggregations) |
| **State** | Maintains checkpoints | Stateless |
| **Best for** | Large data volumes | Aggregated results |

---

**Rule of thumb:**
* **Bronze:** Streaming tables (ingest raw data)
* **Silver:** Streaming tables (incremental cleaning)
* **Gold:** Materialized views (aggregations)

---

### Declarative Pipeline Syntax

**SDP uses Python decorators to declare tables:**

---

### **Basic syntax:**

```python
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# Streaming table
@dp.table(
    name="table_name",
    comment="Description of the table"
)
def table_name():
    return spark.readStream.table("source_table")

# Materialized view
@dp.materialized_view(
    name="view_name",
    comment="Description of the view"
)
def view_name():
    return spark.read.table("upstream_table").groupBy(...).agg(...)
```

---

### **Key components:**

**1. Import pipelines module:**
```python
from pyspark import pipelines as dp
```

**2. Decorator:**
* `@dp.table` - Creates a streaming table
* `@dp.materialized_view` - Creates a materialized view
* `@dp.temporary_view` - Creates a temporary (non-published) view

**3. Function:**
* Function name becomes table name (if not specified)
* Returns a DataFrame
* Can use PySpark transformations

**4. Reading data:**
* `spark.readStream.table()` - Read from source or pipeline table (streaming)
* `spark.read.table()` - Read from source or pipeline table (batch)

---

### **Dependencies are automatic:**

```python
# Bronze
@dp.table
def bronze_bookings():
    return spark.readStream.table("samples.wanderbricks.bookings")

# Silver (depends on bronze)
@dp.table
def silver_bookings():
    return spark.readStream.table("bronze_bookings").filter(...)  # Automatic dependency!

# Gold (depends on silver)
@dp.materialized_view
def gold_revenue():
    return spark.read.table("silver_bookings").groupBy(...).agg(...)  # Automatic dependency!
```

**SDP automatically:**
* Detects dependencies
* Runs tables in correct order
* Handles failures and retries
* Manages incremental processing

---

## Part 2: Bronze Layer - Raw Data Ingestion

The bronze layer ingests raw data using **streaming tables**.

**Bronze characteristics:**
* Ingest data as-is (no transformations)
* Streaming tables for incremental processing
* Append-only
* Full audit trail

---

In [0]:
# Import required modules
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# ============================================
# BRONZE LAYER: Raw Data Ingestion
# ============================================

@dp.table(
    name="bronze_bookings",
    comment="Raw booking data from Wanderbricks - Bronze layer"
)
def bronze_bookings():
    """
    Ingest raw booking data as streaming table.
    No transformations - data as-is from source.
    """
    return (
        spark.readStream
            .table("samples.wanderbricks.bookings")
    )

@dp.table(
    name="bronze_properties",
    comment="Raw property data from Wanderbricks - Bronze layer"
)
def bronze_properties():
    """
    Ingest raw property data as streaming table.
    """
    return (
        spark.readStream
            .table("samples.wanderbricks.properties")
    )

@dp.table(
    name="bronze_reviews",
    comment="Raw review data from Wanderbricks - Bronze layer"
)
def bronze_reviews():
    """
    Ingest raw review data as streaming table.
    """
    return (
        spark.readStream
            .table("samples.wanderbricks.reviews")
    )

### Bronze Layer - Key Points

**What we did:**

1. **Created 3 streaming tables:**
   * `bronze_bookings` - Booking transactions
   * `bronze_properties` - Property listings
   * `bronze_reviews` - Customer reviews

2. **Used `@dp.table` decorator:**
   * Declares a streaming table
   * Automatically managed by SDP
   * Incremental processing

3. **Used `spark.readStream.table()`:**
   * Reads from source tables as streams
   * Processes incrementally
   * Tracks progress automatically

**Why streaming tables for bronze?**
* Incremental ingestion (only new data)
* Low latency
* Efficient for large data volumes
* Automatic checkpoint management
* Append-only semantics

**What SDP handles automatically:**
* Checkpoint management
* Incremental processing
* Schema evolution
* Failure recovery
* Backpressure

---

**Tip:** Bronze layer is always streaming tables - ingest raw data as-is

## Part 3: Silver Layer - Cleaned & Validated Data

The silver layer cleans and validates data using **streaming tables**.

**Silver characteristics:**
* Clean and validate data
* Remove duplicates
* Filter invalid records
* Join related tables
* Still streaming (incremental)

---

In [0]:
# ============================================
# SILVER LAYER: Cleaned and Validated Data
# ============================================

@dp.table(
    name="silver_bookings",
    comment="Cleaned and validated booking data - Silver layer"
)
def silver_bookings():
    """
    Clean bronze bookings:
    - Filter confirmed bookings only
    - Remove invalid data (null amounts, invalid dates)
    - Add calculated columns
    - Deduplicate
    """
    return (
        spark.readStream.table("bronze_bookings")
        # Data quality filters
        .filter(col("status") == "confirmed")
        .filter(col("total_amount").isNotNull())
        .filter(col("total_amount") > 0)
        .filter(col("check_in") < col("check_out"))  # Valid date range
        # Add calculated columns
        .withColumn("nights", datediff(col("check_out"), col("check_in")))
        .withColumn("booking_date", to_date(col("created_at")))
        .withColumn("booking_year", year(col("created_at")))
        .withColumn("booking_month", month(col("created_at")))
        # Select relevant columns
        .select(
            "booking_id",
            "user_id",
            "property_id",
            "check_in",
            "check_out",
            "nights",
            "guests_count",
            "total_amount",
            "status",
            "booking_date",
            "booking_year",
            "booking_month",
            "created_at"
        )
    )

@dp.table(
    name="silver_properties",
    comment="Cleaned property data - Silver layer"
)
def silver_properties():
    """
    Clean bronze properties:
    - Filter valid properties
    - Standardize property types
    """
    return (
        spark.readStream.table("bronze_properties")
        # Data quality filters
        .filter(col("property_type").isNotNull())
        .filter(col("base_price").isNotNull())
        .filter(col("base_price") > 0)
        # Select relevant columns
        .select(
            "property_id",
            "host_id",
            "destination_id",
            "title",
            "property_type",
            "base_price",
            "max_guests",
            "bedrooms",
            "bathrooms",
            "created_at"
        )
    )

@dp.table(
    name="silver_reviews",
    comment="Cleaned review data - Silver layer"
)
def silver_reviews():
    """
    Clean bronze reviews:
    - Filter non-deleted reviews
    - Filter valid ratings
    """
    return (
        spark.readStream.table("bronze_reviews")
        # Data quality filters
        .filter(col("is_deleted") == False)
        .filter(col("rating").isNotNull())
        .filter((col("rating") >= 1.0) & (col("rating") <= 5.0))
        # Add calculated columns
        .withColumn("review_date", to_date(col("created_at")))
        .withColumn("review_year", year(col("created_at")))
        .withColumn("review_month", month(col("created_at")))
        # Select relevant columns
        .select(
            "review_id",
            "booking_id",
            "property_id",
            "user_id",
            "rating",
            "comment",
            "review_date",
            "review_year",
            "review_month",
            "created_at"
        )
    )

### Silver Layer - Key Points

**What we did:**

1. **Created 3 streaming tables:**
   * `silver_bookings` - Cleaned bookings
   * `silver_properties` - Cleaned properties
   * `silver_reviews` - Cleaned reviews

2. **Used `spark.readStream.table()`:**
   * Reads from bronze streaming tables
   * Maintains streaming semantics
   * Incremental processing continues

3. **Applied data quality rules:**
   * Filter invalid data (nulls, negatives, invalid ranges)
   * Remove soft-deleted records
   * Validate business rules

4. **Added calculated columns:**
   * `nights` - Duration of stay
   * `booking_date` - Date portion of timestamp
   * `booking_year`, `booking_month` - For aggregations

**Why streaming tables for silver?**
* Incremental processing (efficient)
* Low latency
* Handles large data volumes
* Maintains data lineage

**Data quality patterns:**
```python
# Filter nulls
.filter(col("column").isNotNull())

# Filter invalid values
.filter(col("amount") > 0)

# Filter by status
.filter(col("status") == "confirmed")

# Validate ranges
.filter((col("rating") >= 1.0) & (col("rating") <= 5.0))
```

---

**Tip:** Silver layer filters bad data and adds business logic

In [0]:
# ============================================
# SILVER LAYER: Enriched Data with JOINs
# ============================================

@dp.table(
    name="silver_bookings_enriched",
    comment="Bookings enriched with property details - Silver layer"
)
def silver_bookings_enriched():
    """
    Join bookings with properties to create enriched dataset.
    This is still a streaming table for incremental processing.
    """
    bookings = spark.readStream.table("silver_bookings")
    properties = spark.readStream.table("silver_properties")
    
    return (
        bookings
        .join(
            properties,
            bookings.property_id == properties.property_id,
            "left"
        )
        .select(
            bookings["*"],
            properties["property_type"],
            properties["title"].alias("property_name"),
            properties["base_price"],
            properties["bedrooms"],
            properties["bathrooms"]
        )
    )

### Silver Layer - JOINs in Streaming

**What we did:**

1. **Created enriched streaming table:**
   * `silver_bookings_enriched` - Bookings + property details

2. **Joined two streaming tables:**
   * Read both as streams with `spark.readStream.table()`
   * Perform stream-stream JOIN
   * Result is still a streaming table

3. **Selected columns from both tables:**
   * All booking columns
   * Property type, name, price, bedrooms, bathrooms

**Stream-stream JOINs:**
* Both sides are streaming
* Incremental processing
* Lakeflow manages state
* Automatic watermarking

**JOIN patterns in pipelines:**
```python
# Stream-stream JOIN
left_stream = spark.readStream.table("table1")
right_stream = spark.readStream.table("table2")
result = left_stream.join(right_stream, "key")

# Stream-batch JOIN (dimension lookup)
stream = spark.readStream.table("fact_table")
dimension = spark.read.table("dimension_table")  # Batch read
result = stream.join(dimension, "key")
```

---

**Tip:** Use stream-stream JOINs for fact tables, stream-batch for dimension lookups

## Part 4: Gold Layer - Aggregated Metrics

The gold layer creates business metrics using **materialized views**.

**Gold characteristics:**
* Aggregated data (GROUP BY)
* Business-ready metrics
* Materialized views (full refresh)
* Optimized for analytics
* Smaller result sets

---

In [0]:
# ============================================
# GOLD LAYER: Aggregated Business Metrics
# ============================================

@dp.materialized_view(
    name="gold_daily_bookings",
    comment="Daily booking metrics - Gold layer"
)
def gold_daily_bookings():
    """
    Daily aggregated booking metrics.
    Uses materialized view for full refresh aggregations.
    """
    return (
        spark.read.table("silver_bookings_enriched")
        .groupBy("booking_date", "property_type")
        .agg(
            count("booking_id").alias("booking_count"),
            sum("total_amount").alias("total_revenue"),
            avg("total_amount").alias("avg_booking_value"),
            sum("nights").alias("total_nights"),
            avg("nights").alias("avg_nights"),
            countDistinct("user_id").alias("unique_customers"),
            countDistinct("property_id").alias("unique_properties")
        )
        .orderBy("booking_date", "property_type")
    )

@dp.materialized_view(
    name="gold_property_performance",
    comment="Property performance metrics - Gold layer"
)
def gold_property_performance():
    """
    Aggregated metrics by property type.
    """
    return (
        spark.read.table("silver_bookings_enriched")
        .groupBy("property_type")
        .agg(
            count("booking_id").alias("total_bookings"),
            sum("total_amount").alias("total_revenue"),
            avg("total_amount").alias("avg_revenue_per_booking"),
            avg("nights").alias("avg_stay_duration"),
            avg("guests_count").alias("avg_guests"),
            countDistinct("property_id").alias("property_count")
        )
        .orderBy(desc("total_revenue"))
    )

@dp.materialized_view(
    name="gold_review_summary",
    comment="Review rating summary by property type - Gold layer"
)
def gold_review_summary():
    """
    Average ratings by property type and time period.
    Joins reviews with properties.
    """
    reviews = spark.read.table("silver_reviews")
    properties = spark.read.table("silver_properties")
    
    return (
        reviews
        .join(properties, "property_id", "left")
        .groupBy("property_type", "review_year", "review_month")
        .agg(
            count("review_id").alias("review_count"),
            avg("rating").alias("avg_rating"),
            min("rating").alias("min_rating"),
            max("rating").alias("max_rating")
        )
        .orderBy("review_year", "review_month", "property_type")
    )

### Gold Layer - Key Points

**What we did:**

1. **Created 3 materialized views:**
   * `gold_daily_bookings` - Daily metrics by property type
   * `gold_property_performance` - Overall property type performance
   * `gold_review_summary` - Review ratings over time

2. **Used `@dp.materialized_view` decorator:**
   * Creates materialized views
   * Full refresh on each update
   * Optimized for aggregations

3. **Used `spark.read.table()` (batch read):**
   * Batch read from silver tables
   * Appropriate for aggregations
   * Full table scan

4. **Applied aggregations:**
   * `count()`, `sum()`, `avg()`, `min()`, `max()`
   * `countDistinct()` for unique counts
   * `groupBy()` for dimensions

**Why materialized views for gold?**
* Aggregations require full data
* Smaller result sets (efficient full refresh)
* Optimized for analytics
* Simpler than streaming aggregations
* Perfect for dashboards

**Streaming table vs Materialized view for gold:**

| Use Case | Recommendation |
|----------|----------------|
| Simple aggregations | Materialized view |
| Large fact tables | Streaming table |
| Real-time metrics | Streaming table |
| Dashboard metrics | Materialized view |
| Complex joins | Materialized view |

---

**Tip:** Gold layer typically uses materialized views for aggregations

   
## Part 5: Creating Your Pipeline with the Lakeflow Pipelines Editor

Now that we've written the code, let's create the pipeline using the **Lakeflow Pipelines Editor** — an IDE built for developing pipelines that combines code editing, pipeline graphs, data previews, and selective execution on a single surface.

---

   
### Step 1: Understand Pipeline File Organization

**Key concept: Folder-based pipelines**

The Lakeflow Pipelines Editor organizes pipeline code in a **folder structure**:

```
pipeline-root/
├── transformations/       # Source code files (Python/SQL)
│   ├── bronze.py          # Bronze layer definitions
│   ├── silver.py          # Silver layer definitions
│   └── gold.py            # Gold layer definitions
├── explorations/          # Notebooks for viewing output
│   └── explore.py
└── README.md              # Documentation
```

**What this means:**
* Pipeline source code lives in `.py` or `.sql` files (not notebooks)
* You can mix Python and SQL files in a single pipeline
* Code in one file can reference tables defined in another file
* The `transformations` folder is the default location for source code
* Markdown files can be included for documentation

**For this demo:**
* We will create a new pipeline and paste our code into the editor
* Alternatively, you can select **"Add existing assets"** to link existing files

---

**Tip:** The Lakeflow Pipelines Editor is the recommended way to develop pipelines

   
### Step 2: Create a New ETL Pipeline

**Follow these steps:**

1. Click **"New"** in the top of the left sidebar
2. Select **"ETL pipeline"**
3. The Lakeflow Pipelines Editor opens on the create page

**Pipeline creation form:**

4. **Pipeline name:** Click the header to give your pipeline a name (e.g., "Wanderbricks Analytics Pipeline")
5. **Default catalog & schema:** Just below the name, choose the default catalog and schema for your output tables (e.g., `workspace.default`)
6. **Next step for your pipeline** — choose one of:
   * **Start with sample code in SQL** — Creates folder structure with SQL sample code
   * **Start with sample code in Python** — Creates folder structure with Python sample code
   * **Start with a empty file** — Creates folder structure with a blank code file
   * **Add existing assets** — Links existing workspace files to the pipeline

**For this demo:** Select **"Start with a empty file"** (Python), then paste the pipeline code from this notebook into the editor.

**Default settings applied automatically:**
* Unity Catalog enabled
* Current channel
* Serverless compute
* Development mode off (for scheduled runs)

7. You are redirected to the newly created pipeline in the editor

**Checkpoint:** Pipeline created in the Lakeflow Pipelines Editor

---

**Tip:** You can also create a pipeline from the Workspace browser (right-click a folder → Create → ETL pipeline) or from the Jobs & Pipelines page

   
### Pipeline Configuration - Key Settings

**Pipeline mode:**

**Triggered mode:**
* Runs on-demand or on schedule
* Processes all available data
* Stops after completion
* Use for: Batch pipelines, scheduled jobs

**Continuous mode:**
* Runs continuously
* Processes data as it arrives
* Always running
* Use for: Real-time pipelines, streaming data

**For this demo:** Use **Triggered** mode

---

**Accessing settings in the Lakeflow Pipelines Editor:**

* Click the **Settings** button in the pipeline toolbar (top of editor)
* Or click the **gear icon** in the pipeline assets browser (left sidebar)

**Settings panel includes:**
* **General** — Name, default catalog, default schema
* **Source code** — Root folder and source code file configuration
* **Compute** — Serverless (recommended), Enhanced autoscaling, or Fixed size
* **Schedule** — Create one or more schedules (creates a job automatically)
* **Notifications** — Alert configuration
* **Advanced** — Channel, Photon, custom configuration

---

**Default catalog & schema:**
* Where tables will be created when not explicitly qualified in code
* Example: `workspace.default`
* All pipeline tables appear here
* Can query tables after pipeline runs

---

**Compute:**
* **Serverless** — Recommended, no cluster management, fastest startup
* **Enhanced autoscaling** — Automatically scales workers
* **Fixed size** — Specific number of workers

**For this demo:** **Serverless** (default)

---

   
### Step 3: Explore the Lakeflow Pipelines Editor UI

**Follow these steps:**

After creating the pipeline, you land in the editor. Familiarize yourself with the key areas:

**1. Pipeline Assets Browser (left sidebar):**
* Shows all files and folders in your pipeline
* **Pipeline tab** — Files associated with this pipeline
* **All files tab** — Browse all workspace files
* Shortcuts to Settings, Schedule, and Permissions
* Graphical view of recent runs

**2. Multi-file Code Editor (center):**
* Open multiple source files in tabs
* Edit Python (`.py`) or SQL (`.sql`) files
* Supports both languages in the same pipeline

**3. Pipeline Toolbar (top):**
* Pipeline name and configuration options
* **Run pipeline** button — Runs all source code files
* **Run file** — Refreshes only tables in the current file
* **Settings**, **Schedule**, **Share** buttons

**4. Interactive Pipeline Graph (right panel):**
* Toggle with the graph icon in the right panel
* Shows table dependency DAG after running/validating
* Click nodes for data preview and table details

**5. Data Preview (bottom panel):**
* Inspect streaming table and materialized view data
* Appears when you click a node in the graph

**6. Issues Panel:**
* Summarizes errors across all files
* Navigate directly to the error location

**Checkpoint:** Familiar with the Lakeflow Pipelines Editor layout

---

**Tip:** You can manage settings, schedule, and permissions all from the editor toolbar

## Part 6: Running Your Pipeline

Let's execute the pipeline and see it in action!

---

   
### Step 4: Run the Pipeline

**Follow these steps:**

1. In the Lakeflow Pipelines Editor toolbar, click **"Run pipeline"** (top-right)
2. The pipeline begins execution
3. You'll see:
   * Status indicators in the toolbar
   * The pipeline graph populates with nodes
   * Nodes change state as they execute

**Run options available:**

| Action | What it does |
|--------|-------------|
| **Run pipeline** | Runs all source code files in the pipeline |
| **Run file** | Refreshes only the tables defined in the currently open file |
| **Run single table** | Hover a node in the graph and click refresh, or right-click a node |

**What happens when you run:**
* SDP reads all source code files in the pipeline
* Parses all `@dp.table` and `@dp.materialized_view` declarations
* Builds dependency graph automatically
* Provisions serverless compute
* Executes tables in correct dependency order
* Creates tables in the default catalog/schema

**First run:**
* Takes longer (initial data load)
* Creates all tables from scratch
* Processes all historical data

**Subsequent runs:**
* Faster (incremental processing for streaming tables)
* Only processes new data
* Updates existing tables

**Selective execution for development:**
* Use **Run file** to iterate on a single file without running the entire pipeline
* Use **Run single table** (from the graph) to refresh just one table
* This enables step-by-step development and debugging

**Checkpoint:** Pipeline is running

---

**Tip:** Use selective execution during development for faster iteration

   
### Step 5: Explore the Interactive Pipeline Graph (DAG)

**Follow these steps:**

1. The **interactive DAG** appears in the right panel of the editor
2. Toggle it on/off with the **graph icon** in the right side panel
3. You can also maximize it for a full-screen view
4. Watch nodes change state as pipeline runs
5. Click on a completed node to see:
   * Data preview (sample rows)
   * Table schema
   * Row count
   * Execution time
6. Toggle between vertical and horizontal layouts

**What the graph shows:**

**Nodes (tables/views):**
* **Bronze tables** — Raw data sources
* **Silver tables** — Cleaned data
* **Gold views** — Aggregated metrics
* Each node shows table name and current status

**Edges (dependencies):**
* **Arrows** show data flow direction
* Bronze → Silver → Gold
* Automatic dependency detection from your code

**Node states during pipeline lifecycle:**
* **Green** — Completed successfully
* **Blue** — Currently running
* **Yellow** — Queued/waiting
* **Red** — Error
* **Gray** — Not started / Validated

**Interactive features:**
* **Click** a node → Opens data preview and table definition in the bottom panel
* **Hover** a node → Shows toolbar with options (refresh, etc.)
* **Right-click** a node → Context menu with same options
* **Zoom** in/out with controls at bottom-right
* **Layout** — Switch between vertical and horizontal via More options
* When editing a file, its tables are **highlighted** in the graph

**Checkpoint:** Can view and interact with the pipeline DAG

---

**Tip:** The interactive DAG updates in real-time and highlights tables from the file you're editing

   
### Step 6: Monitor Pipeline Execution

**Follow these steps:**

While the pipeline runs:

1. **Watch the interactive graph:**
   * Nodes change state (validated → running → completed)
   * Progress flows from bronze → silver → gold

2. **Check execution insights:**
   * Click on a running/completed node in the graph
   * The bottom panel shows:
     - **Data preview** — Sample data from the table
     - **Execution insights** — Duration, rows processed
     - **Table definition** — Schema and metadata
   * You can view insights for all tables or a single table

3. **View the Issues panel:**
   * Summarizes errors across all files in the pipeline
   * Navigate directly to where errors occurred in your code
   * Complements the error indicators shown inline in the editor

4. **Access the monitoring page (for historical runs):**
   * Click **Jobs & Pipelines** in the left sidebar
   * Or click the run results in the pipeline assets browser
   * The monitoring page shows historical runs (useful for scheduled pipelines)
   * From monitoring, click **"Edit pipeline"** to return to the editor

**Pipeline execution order:**
```
1. Bronze tables (parallel)
   ↓
2. Silver tables (after bronze completes)
   ↓
3. Silver enriched (after silver completes)
   ↓
4. Gold views (after silver completes)
```

**Execution states:**
* **Queued** — Waiting for dependencies
* **Running** — Currently processing
* **Completed** — Successfully finished
* **Failed** — Error occurred

**Checkpoint:** Pipeline execution monitored

---

**Tip:** The Issues panel is your best friend for debugging — it pinpoints errors across all files

   
### 📊 Step 7: View Pipeline Results

**Follow these steps:**

After pipeline completes:

1. **Check final status:**
   * All nodes in the graph should be green
   * The toolbar shows "Succeeded"
   * Total execution time is displayed

2. **View data previews in the editor:**
   * Click on any node in the interactive DAG
   * The **data preview** panel opens at the bottom of the editor
   * Shows sample rows from each streaming table and materialized view
   * View schema, row count, and execution insights

3. **Query the tables from any SQL editor or notebook:**
   * Tables are created in your default catalog/schema
   * Open a SQL query or notebook
   * Query the tables:
   ```sql
   -- Bronze
   SELECT * FROM workspace.default.bronze_bookings LIMIT 10;
   
   -- Silver
   SELECT * FROM workspace.default.silver_bookings LIMIT 10;
   
   -- Gold
   SELECT * FROM workspace.default.gold_daily_bookings;
   ```

4. **Verify data flow:**
   * Bronze has raw data (all records)
   * Silver has cleaned data (fewer rows due to filters)
   * Gold has aggregated data (much fewer rows)

**Checkpoint:** Can preview data in the editor and query pipeline output tables

---

**Tip:** Data previews in the editor let you inspect results without leaving the pipeline context

## Practice Challenges

Now it's your turn! Complete these challenges to reinforce your learning.

Create a new ETL Pipeline and complete each of these challenges in a seperate file.

---

### Challenge 1: Add a New Bronze Table

**Your task:**

Add a bronze table for the users data.

**Requirements:**
1. Create a streaming table named `bronze_users`
2. Ingest from `samples.wanderbricks.users`
3. Add appropriate comment
4. Use `@dp.table` decorator
5. Use `spark.readStream.table()`

**Expected code structure:**
```python
@dp.table(
    name="bronze_users",
    comment="Your comment here"
)
def bronze_users():
    return spark.readStream.table("samples.wanderbricks.users")
```

**After adding:**
1. Go to pipeline UI
2. Click "Run Pipeline" to run
3. Verify new table appears in graph

**Success criteria:** bronze_users table created and visible in pipeline graph

---

### Challenge 2: Create Silver Users Table

**Your task:**

Create a cleaned silver table for users.

**Requirements:**
1. Create streaming table named `silver_users`
2. Read from `bronze_users` using `spark.readStream.table()`
3. Apply data quality filters:
   * Filter out null emails
   * Filter out null names
4. Add calculated column:
   * `account_age_days` = DATEDIFF(CURRENT_DATE(), created_at)
5. Select relevant columns

**Hint:**
```python
@dp.table(
    name="silver_users",
    comment="Cleaned user data - Silver layer"
)
def silver_users():
    return (
        spark.readStream.table("bronze_users")
        .filter(col("email").isNotNull())
        .filter(col("name").isNotNull())
        .withColumn("account_age_days", datediff(current_date(), col("created_at")))
        .select("user_id", "email", "name", "country", "user_type", 
                "created_at", "account_age_days")
    )
```

**After adding:**
1. Go to pipeline UI
2. Click "Run Pipeline" to run
3. Verify new table appears in graph
4. Check it connects to bronze_users

**Success criteria:** silver_users table with data quality filters

---

### Challenge 3: Create Gold Metrics View

**Your task:**

Create a gold materialized view with user metrics.

**Requirements:**
1. Create materialized view named `gold_user_metrics`
2. Use `@dp.materialized_view` decorator
3. Read from `silver_users` using `spark.read.table()`
4. Group by `country` and `user_type`
5. Calculate:
   * Total user count
   * Average account age
   * Business user count
6. Order by total users descending

**Hint:**
```python
@dp.materialized_view(
    name="gold_user_metrics",
    comment="User metrics by country and type - Gold layer"
)
def gold_user_metrics():
    return (
        spark.read.table("silver_users")
        .groupBy("country", "user_type")
        .agg(
            count("user_id").alias("total_users"),
            avg("account_age_days").alias("avg_account_age_days"),
            sum(when(col("is_business") == True, 1).otherwise(0)).alias("business_users")
        )
        .orderBy(desc("total_users"))
    )
```

**After adding:**
1. Go to pipeline UI
2. Click "Run Pipeline" to run
3. Verify new table appears in graph
4. Query the view to see results

**Success criteria:** gold_user_metrics view with aggregated data

---

## Querying Your Pipeline Results

**After pipeline completes, query the tables:**

### **Bronze tables:**
```sql
-- Raw data (all records)
SELECT COUNT(*) FROM workspace.default.bronze_bookings;
SELECT * FROM workspace.default.bronze_bookings LIMIT 10;
```

### **Silver tables:**
```sql
-- Cleaned data (fewer records)
SELECT COUNT(*) FROM workspace.default.silver_bookings;
SELECT * FROM workspace.default.silver_bookings_enriched LIMIT 10;
```

### **Gold views:**
```sql
-- Aggregated metrics
SELECT * FROM workspace.default.gold_daily_bookings 
ORDER BY booking_date DESC 
LIMIT 30;

SELECT * FROM workspace.default.gold_property_performance;

SELECT * FROM workspace.default.gold_review_summary 
WHERE review_year = 2024;
```

---

### **Verify data quality:**
```sql
-- Compare row counts
SELECT 'Bronze' AS layer, COUNT(*) AS row_count 
FROM workspace.default.bronze_bookings
UNION ALL
SELECT 'Silver' AS layer, COUNT(*) AS row_count 
FROM workspace.default.silver_bookings
UNION ALL
SELECT 'Gold' AS layer, COUNT(*) AS row_count 
FROM workspace.default.gold_daily_bookings;
```

**Expected results:**
* Bronze: Most rows (all data)
* Silver: Fewer rows (filtered)
* Gold: Much fewer rows (aggregated)

---

**Tip:** Use these tables in dashboards, queries, and downstream applications

## Troubleshooting Common Issues

**Pipeline won't start:**
* Check notebook is saved
* Verify notebook path in pipeline settings
* Ensure target schema exists
* Check permissions on source tables
* Verify SQL warehouse or cluster is available

**Table shows as failed (red):**
* Click the node to see error message
* Check Event log for details
* Common issues:
   - Syntax errors in code
   - Invalid column references
   - Source table doesn't exist
   - Permission issues
* Fix code and rerun

**Dependencies not detected:**
* Use `spark.read.table()` or `spark.readStream.table()` to reference pipeline tables
* Check table names match exactly
* Verify decorator syntax is correct

**Streaming table not processing incrementally:**
* Verify using `readStream` (not `read`)
* Check source supports streaming
* Ensure checkpoint location is accessible

**Materialized view taking too long:**
* Check if aggregating too much data
* Add filters to reduce data volume
* Consider using streaming table instead
* Optimize source queries

**Schema evolution issues:**
* SDP handles schema evolution automatically
* Check if column types are compatible
* Review schema in table details

---

**Tip:** Event log is your best friend for debugging - shows detailed error messages

## Congratulations!

You've completed the Lakeflow SDP Pipeline Fundamentals demo!

### **What You Learned:**

* **SDP concepts** - Declarative pipeline framework  
* **Streaming tables** - Incremental processing with `@dp.table`  
* **Materialized views** - Full refresh aggregations with `@dp.materialized_view`  
* **Bronze layer** - Raw data ingestion  
* **Silver layer** - Data cleaning and validation  
* **Gold layer** - Business metrics  
* **Pipeline creation** - UI configuration  
* **Pipeline execution** - Running and monitoring  
* **Data lineage** - Visual graph  
* **Best practices** - Production patterns  

---

### **Key Takeaways:**

1. **Declarative is simpler** - Define what, not how
2. **Streaming tables for incremental** - Bronze and silver
3. **Materialized views for aggregations** - Gold layer
4. **Dependencies are automatic** - No manual orchestration
5. **Graph shows lineage** - Visual data flow
6. **Incremental by default** - Efficient processing
7. **Production-ready** - Built-in reliability

---

### **Next Steps:**

* Build pipelines for your own data
* Explore expectations for data quality
* Learn about CDC patterns
* Set up production pipelines
* Create dashboards from gold tables
* Schedule pipeline runs
* Monitor pipeline health

---

### **Resources:**

* [Lakeflow SDP Concepts](https://docs.databricks.com/aws/en/ldp/concepts/)
* [Lakeflow Spark Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/)
* [Python Language Reference](https://docs.databricks.com/aws/en/ldp/developer/python-ref/)
* [Develop Pipeline Code](https://docs.databricks.com/aws/en/ldp/developer/python-dev/)
* [Transform Data with Pipelines](https://docs.databricks.com/aws/en/ldp/transform/)